In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya
import pandas as pd

In [ ]:
import ana_utils

In [ ]:
import sys
sys.path.append("../photometry")
sys.path.append("../imaging/")
import phot_utils

In [ ]:
from astropy import units as u

In [ ]:
from dataclasses import dataclass

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
import ana_utils

In [ ]:
from astropy.table import Table

In [ ]:
import filter_utils

In [ ]:
from plot_utils import plot_mag_comparison

# Yasone 1

In [ ]:
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")


In [ ]:
# cat_sep = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="julen")


In [ ]:
cat_julen = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_julen_stack")


In [ ]:
cat_lss = Table.read("../survey_data/yasone1_lss10.csv")

In [ ]:
cat_ps = Table.read("../survey_data/yasone1_panstarrs.fits")

In [ ]:
bins = np.linspace(15, 30, 100)
plt.hist(cat_lss["mag_r"], bins, histtype="step")
plt.hist(cat["R_MAG"], bins, histtype="step")
plt.hist(cat_ps["rMeanApMag"], bins, histtype="step")

In [ ]:
bins = np.linspace(15, 30, 100)
plt.hist(cat["G_MAG"], bins, histtype="step")

plt.hist(cat_lss["mag_g"], bins, histtype="step")
plt.hist(cat_ps["gMeanApMag"], bins, histtype="step")

In [ ]:
np.sum(np.isfinite(cat_lss["mag_g"])), np.sum(np.isfinite(cat_lss["mag_r"])), np.sum(np.isfinite(cat_lss["mag_i"]))

In [ ]:
for col in cat_ps.colnames:
    cat_ps[col] = cat_ps[col].reshape(-1)

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
cat_joined = phot_utils.outer_join_xmatch(cat_joined, cat_ps, ra1="ra", dec1="dec", ra2="ra", dec2="dec")



In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MAG", mag_cmp_fmt="mag_{filt}", 
                    mag_ref_err_fmt="{filt}_MAG_ERR", mag_cmp_err_fmt="mag_err_{filt}", 
                    ref_mag_upper=True, cmp_mag_upper=False, 
                    filters=["g", "r"],
                    ref_label="osiris",
                    cmp_label="LSS"
                    
                   )

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MAG", mag_cmp_fmt="{filt}MeanPSFMag", 
                    mag_ref_err_fmt="{filt}_MAG_ERR", mag_cmp_err_fmt="{filt}MeanPSFMagErr", 
                    ref_mag_upper=True, cmp_mag_upper=False, 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
import astropy

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_sep, ra1="R_ALPHA_J2000", dec1="R_DELTA_J2000", ra2="R_ALPHA_J2000", dec2="R_DELTA_J2000")


In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MAG_1", mag_cmp_fmt="{filt}_MAG_2", 
                    mag_ref_err_fmt="{filt}_MAG_ERR_1", mag_cmp_err_fmt="{filt}_MAG_ERR_2", 
                    ref_mag_upper=True, cmp_mag_upper=True, filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18), )

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_julen, ra1="R_ALPHA_J2000", dec1="R_DELTA_J2000", ra2="R_ALPHA_J2000", dec2="R_DELTA_J2000")


In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MAG_1", mag_cmp_fmt="{filt}_MAG_2", 
                    mag_ref_err_fmt="{filt}_MAG_ERR_1", mag_cmp_err_fmt="{filt}_MAG_ERR_2", 

                    ref_mag_upper=True, cmp_mag_upper=True, filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18), )

## Photometry ssystems

In [ ]:
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")

cat_small_bkg = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_small_bkg", shiftname="allcolours_panstarrs_shift")
cat_large_bkg = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_large_bkg", shiftname="allcolours_panstarrs_shift")
cat_small_bkg_filt = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_small_bkg_filt", shiftname="allcolours_panstarrs_shift")
cat_large_bkg_filt = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_large_bkg_filt", shiftname="allcolours_panstarrs_shift")


In [ ]:
for tab in [cat, cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    filter_utils.calibrate_mag_col(tab, "MED_MAG_APER_3")

In [ ]:
cat["G_aperture_sum_0

In [ ]:
for cat2 in [cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    fig, axs = plt.subplots(1, 3, figsize=(6, 2), sharex=True, sharey=True)

    cmax = 10_000
    plt.sca(axs[0])
    plt.scatter(cat["ra"], cat["dec"], c=cat["G_aperture_sum_3_median"] - cat2["G_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)

    plt.sca(axs[1])
    plt.scatter(cat["ra"], cat["dec"], c=cat["R_aperture_sum_3_median"] - cat2["R_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)

    plt.sca(axs[2])
    plt.scatter(cat["ra"], cat["dec"], c=cat["I_aperture_sum_3_median"] - cat2["I_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)
    plt.colorbar(label="flux difference")

In [ ]:
for cat2 in [cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    cat_joined = phot_utils.outer_join_xmatch(cat, cat2, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
    
    plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MED_MAG_APER_3_1", mag_cmp_fmt="{filt}_MED_MAG_APER_3_2", 
                        mag_ref_err_fmt="{filt}_MED_MAG_APER_3_ERR_1", mag_cmp_err_fmt="{filt}_MED_MAG_APER_3_ERR_2", 
    
                        ref_mag_upper=True, cmp_mag_upper=True, filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18))
    plt.show()

# Yasone 2

In [ ]:
cat = ana_utils.read_catalogue("yasone2", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")


In [ ]:
cat_lss = Table.read("../survey_data/yasone2_lss10.csv")

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="R_ALPHA_J2000", dec1="R_DELTA_J2000", ra2="ra", dec2="dec")

In [ ]:
np.sum(np.isfinite(cat_lss["mag_g"])), np.sum(np.isfinite(cat_lss["mag_r"])), np.sum(np.isfinite(cat_lss["mag_i"]))

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MAG", mag_cmp_fmt="mag_{filt}", ref_mag_upper=True, cmp_mag_upper=False, filters=["g", "i"])

In [ ]:
cat = ana_utils.read_catalogue("yasone3", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")


In [ ]:
cat_lss = Table.read("../survey_data/yasone3_lss10.csv")

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="R_ALPHA_J2000", dec1="R_DELTA_J2000", ra2="ra", dec2="dec")

In [ ]:
np.sum(np.isfinite(cat_lss["mag_g"])), np.sum(np.isfinite(cat_lss["mag_r"])), np.sum(np.isfinite(cat_lss["mag_i"]))

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MAG", mag_cmp_fmt="mag_{filt}", ref_mag_upper=True, cmp_mag_upper=False, filters=["g", "i"])